# 📓 MapBiomas Fire Monitor — Pipeline (M0-M7)

## 📘 Documentación y Guía del Usuario
Expanda esta sección para entender la arquitectura del dato, requisitos de ambiente y flujo de trabajo. 

#### 📌 Introducción y Contexto de Uso

Este pipeline de código constituye el núcleo de procesamiento automatizado para el **Monitoramiento de Cicatrices de Fuego de MapBiomas**. Su objetivo principal es extraer, procesar, clasificar y publicar datos satelitales a nivel regional o nacional con alta trazabilidad.

*   📥 **Entradas Globales (Inputs):** Colecciones de imágenes brutas desde Google Earth Engine (Sentinel-2, Landsat), polígonos vectoriales para entrenamiento (muestras / *samples*), y mapas auxiliares de cobertura de suelo (LULC).
*   📤 **Salidas Globales (Outputs):** Chunks e mosaicos optimizados (COGs) en Google Cloud Storage (GCS), pesos de modelos de redes neuronales (DNN), matrices de classificação, e ImageCollections versionadas pre-oficiales en GEE.

> **Nota:** Tanto el disco del PC local como el almacenamiento temporal de Colab actúan como **espacio efímero**. La persistencia ocurre siempre en el **Google Cloud Storage (GCS) Bucket**.

#### 🚦 Ciclo de Vida del Dato y Reglas de Nomenclatura

| Etapa | Peso | Inputs → Outputs | Regla de Nomenclatura | Ejemplo |
| :--- | :--- | :--- | :--- | :--- |
| **M1: Export** | **Leve** | **IN:** Colecciones GEE<br>**OUT:** Chunks GCS | `image_{country}_fire_{sensor}_{mosaic}_{band}_{temporal_id}_{suffix}` | `image_peru_fire_sentinel2_minnbr_blue_2025_08_00000-00000` |
| **M2: Mosaic** | **Medio** | **IN:** Chunks GCS<br>**OUT:** Mosaicos COG (GCS) | Semilla M1 + sufijo `_cog` | `image_peru_fire_sentinel2_minnbr_blue_2025_08_cog` |
| **M3: Samples** | **Leve** | **IN:** Mosaicos (M2)<br>**OUT:** Polígonos (GCS/Asset) | `sample_{id}_{country}_{region}_{temporal_id}`| `sample_0001_peru_r10_amazon_2025_07` |
| **M4: Train** | **Medio** | **IN:** Muestras M3 + Mosaicos M2<br>**OUT:** DNN (GCS) | `training_{id}_{region}_{sensor}` | `training_0001_peru_r10_sentinel2` |
| **M5: Classify**| **Pesado**| **IN:** Modelo M4 + Mosaicos M2<br>**OUT:** Raster (GCS) | `region_{reg}_training_{id}_{sensor}_{temp_id}`| `region_r10_training_0001_sentinel2_2025_08` |

### 📂 Arquitectura de Datos e Relacionamento (M1-M7)

#### 🧭 Mapa de Persistência (Onde encontrar os dados)

| Etapa | Extensão | Path Principal no Cloud Storage (GCS) | 
| :--- | :--- | :--- |
| **M1: Export** | `.tif` | `library_images/{sensor}/{period}/{mosaic}/{temporal_id}/chunks/` |
| **M2: Mosaic** | `.tif` | `library_images/{sensor}/{period}/{mosaic}/{temporal_id}/` |
| **M3: Samples** | `.csv` | `library_samples/` |
| **M4: Train** | `.pb / .json` | `library_images/{sensor}/models/training_{training_id}_{shortname}_{sensor}/` |
| **M5: Classify** | `.tif` | `library_images/{sensor}/{period}/burned_day_of_year_regional/` |

#### 🏷️ Regras de Nomenclatura Padrão
*   **Imagens (M1/M2):** `image_{country}_fire_{sensor}_{mosaic}_{band}_{temporal_id}`
*   **Amostras (M3):** `sample_{id}_{country}_{region}_{temporal_id}`
*   **Classificação (M5):** `region_{region}_training_{training_id}_{sensor}_{temporal_id}`

---

## ⚙️ [M0] — Configuración de Ambiente (Escolha uma Rota)

### > Opción A: Inicialización Google Colab
**💡 Nota para Colab:** Las siguientes celdas preparan el entorno virtual en la nube.

In [ ]:
# M0.1a — Preparación del entorno Colab (Clonar repo)
import os
if not os.path.exists("fire_monitor"):
    !git clone https://github.com/mapbiomas/peru-fire.git fire_monitor
%cd fire_monitor/mapbiomas_fire_monitor/version_01/src/core

In [ ]:
# ⬇️ Instalar GDAL Binaries e dependências Python
!apt-get update -qq && apt-get install -y -qq gdal-bin python3-gdal
!pip install -q earthengine-api gcsfs rasterio scipy tqdm

### > Opción B: Inicialización Local
**🛠️ Requisitos Local (GDAL / Conda)**
Si recibes un error de `Faltam dependências vitais`, el monitor buscará automáticamente el GDAL en tu instalación de Conda.

In [ ]:
# M0.1b — Configuración local de rutas
import sys, os
REPO_ROOT = os.path.abspath("..")
SRC_PATH  = os.path.join(REPO_ROOT, "src", "core")
if SRC_PATH not in sys.path: sys.path.insert(0, SRC_PATH)

### > Inicialización Común (M0.2)
**💡 Célula "Invencível":** Esta célula autodetecta se estás en Local ou Colab y configura as rutas.

In [ ]:
# M0.2 — Inicialização do Monitor
import sys, os

def auto_path_setup():
    """Localiza a pasta src/core em diferentes ambientes"""
    possible_paths = [
        os.path.abspath("."),             
        os.path.abspath("../src/core"),   
        "/content/fire_monitor/mapbiomas_fire_monitor/version_01/src/core", 
    ]
    for p in possible_paths:
        if os.path.exists(os.path.join(p, "M0_auth_config.py")):
            if p not in sys.path: sys.path.insert(0, p)
            return p
    return None

found_path = auto_path_setup()
COUNTRY = "peru"

from M0_auth_config import set_country, authenticate, set_global_opts, print_config
set_country(COUNTRY)
set_global_opts(sensor='sentinel2', periodicity='monthly', personal_task_flag='MONITOR', clean_cache=False)
authenticate() 
print_config()

## [M1] — Despacho de Exportación (GEE → Bucket)

In [ ]:
from M1_export_ui import run_ui, start_export
ui_exporter = run_ui(years=[2025, 2026])

In [ ]:
# Gatilho de Exportação (GEE -> GCS)
start_export(ui_exporter)

## [M2] — Ensamblaje Nacional (COG)

In [ ]:
from M2_mosaic_ui import run_ui, start_mosaic_assembly
ui_assembler = run_ui(years=[2025, 2026])

In [ ]:
# Gatilho de Montagem (COG Local/GCS)
start_mosaic_assembly(ui_assembler)

## [M3] — Coleta de Amostras (GEE Toolkit Gateway)

In [ ]:
from M3_toolkit_ui import show_toolkit_links
show_toolkit_links()

## [M4] — Entrenamiento DNN

In [ ]:
from M4_model_trainer import run_ui, start_training
ui_trainer = run_ui()

In [ ]:
# Gatilho de Treinamento
start_training(ui_trainer)

## [M5] — Clasificación Regional

In [ ]:
from M5_classifier import run_ui, start_classification
ui_classifier = run_ui()
# start_classification(ui_classifier)

## [M6-M7] — Repensar Pipeline (Filtros e Versionamento)
Esta seção será atualizada conforme as novas definições de pós-processamento.

In [ ]:
print("Módulos M6 e M7 em fase de reformulação.")